# BIKV v17 Stage H controlled chains 4k

Frozen exploratory Stage H. It extends the adjacent-boundary experiment to arbitrary block pairs and multi-block chains. Re-running the execution cell resumes from mirrored result rows.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
import torch
IS_KAGGLE = Path('/kaggle/input').exists() and Path('/kaggle/working').exists()
IS_COLAB = Path('/var/colab/hostname').exists() and not IS_KAGGLE
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU accelerator before continuing.')
gpu = torch.cuda.get_device_properties(0)
print('Platform:', 'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'local')
print('GPU:', gpu.name, '| VRAM GiB:', round(gpu.total_memory / 2**30, 1))


In [ ]:
WORK = Path('/content/bikv_v17_stage_h_chain_4k_work') if IS_COLAB else Path('/kaggle/working/bikv_v17_stage_h_chain_4k_work')
if IS_COLAB:
    from google.colab import drive, files
    drive.mount('/content/drive')
    MIRROR = Path('/content/drive/MyDrive/bikv_runs')
    candidates = list(Path('/content').glob('boundary_interaction_kv_v17_stage_h*.zip'))
    if not candidates:
        files.upload()
        candidates = list(Path('/content').glob('boundary_interaction_kv_v17_stage_h*.zip'))
elif IS_KAGGLE:
    MIRROR = None
    candidates = list(Path('/kaggle/input').rglob('boundary_interaction_kv_v17_stage_h*.zip'))
    candidates += list(Path('/kaggle/working').glob('boundary_interaction_kv_v17_stage_h*.zip'))
else:
    MIRROR = None
    candidates = list(Path.cwd().glob('boundary_interaction_kv_v17_stage_h*.zip'))
SOURCE = WORK / 'source_v17'
if SOURCE.exists():
    shutil.rmtree(SOURCE)
SOURCE.mkdir(parents=True, exist_ok=True)
if candidates:
    archive_path = max(candidates, key=lambda path: path.stat().st_mtime)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(SOURCE)
elif IS_KAGGLE:
    projects = [p.parent for p in Path('/kaggle/input').rglob('pyproject.toml') if (p.parent / 'src' / 'bikv').is_dir()]
    if len(projects) != 1:
        raise FileNotFoundError(f'Expected one extracted Stage H project, found {projects}')
    shutil.copytree(projects[0], SOURCE / 'boundary_interaction_kv', dirs_exist_ok=True)
else:
    raise FileNotFoundError('Upload boundary_interaction_kv_v17_stage_h.zip')
projects = list(SOURCE.rglob('pyproject.toml'))
if len(projects) != 1:
    raise RuntimeError(f'Expected one project, found {len(projects)}')
PROJECT = projects[0].parent
print('Project:', PROJECT)


In [ ]:
os.environ['HF_HOME'] = str(WORK / 'hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT), 'nltk>=3.9,<4'], check=True)
if sys.version_info >= (3, 13):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'fire==0.7.1'], check=True)
env = {**os.environ, 'PYTHONPATH': f"{PROJECT / 'src'}:{PROJECT / 'scripts'}"}
def stream(command):
    print(' '.join(map(str, command)), flush=True)
    process = subprocess.Popen(command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)
print('Installed frozen Stage H source.')


## Frozen controlled test

36 stories in 18 counterfactual families, balanced across 2/3/4-hop chains and adjacent/dispersed evidence. Budget is six of 42 document blocks. The learned pair model and matched unary model share the same candidate masks, Ridge target, regularization and frozen training data.


In [ ]:
base = json.loads((PROJECT / 'configs' / 'v17_stage_h_synthetic_4k.json').read_text())
base['output_root'] = str(WORK / 'runs')
base['mirror_root'] = str(MIRROR) if MIRROR else None
base['stage_h_chain_data'] = str(PROJECT / 'data' / 'stage_h' / 'stage_h_chains_4k.jsonl')
base['data_path'] = base['stage_h_chain_data']
base['stage_h_chain_audit'] = str(PROJECT / 'data' / 'stage_h' / 'stage_h_audit.json')
base['stage_h_frozen_models_path'] = str(PROJECT / 'configs' / 'v17_stage_h_frozen_models.json')
runtime_path = WORK / 'runtime_stage_h_chain_4k.json'
runtime_path.parent.mkdir(parents=True, exist_ok=True)
runtime_path.write_text(json.dumps(base, indent=2), encoding='utf-8')
print(runtime_path.read_text())


## Run or resume

This cell prints progress after every method. Re-run it after a disconnect; completed JSON rows are restored from Drive in Colab or retained by a Kaggle saved version.


In [ ]:
stream([sys.executable, '-u', str(PROJECT / 'scripts' / 'stage_h_graph.py'), '--config', str(runtime_path)])


In [ ]:
run_root = Path(base['output_root']) / base['run_id']
stream([sys.executable, str(PROJECT / 'scripts' / 'analyze_stage_h.py'), str(run_root), '--seed', str(base['seed']), '--budget', str(base['budget_blocks'])])
sys.path.insert(0, str(PROJECT / 'src'))
from bikv.storage import ResultStore
store = ResultStore(base['output_root'], base['run_id'], base.get('mirror_root'))
bundle = store.bundle()
print('STATUS:', (run_root / 'status.json').read_text())
print('REPORT:', (run_root / 'analysis' / 'report.md').read_text())
print('Artifact:', bundle)
if IS_COLAB:
    from google.colab import files
    files.download(str(bundle))
else:
    print('Kaggle: artifact is in /kaggle/working. Use Save Version to retain it.')
